In [1]:
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path

In [30]:
EMOTIONS_FILE = Path("../data/processed/all_emotions_consolidated.csv")
CPS_FILE = Path("../data/processed/emotion_change_points.csv")
METADATA_FILE = Path("../data/raw/metadata.csv")
OUTPUT_FILE = Path("../data/processed/dataset.csv")

emotions_df = pd.read_csv(EMOTIONS_FILE)
change_points_df = pd.read_csv(CPS_FILE)
metadata_df = pd.read_csv(METADATA_FILE)

In [31]:
import numpy as np
import ast

def change_point_mask(change_points):
    length = 100
    change_points = ast.literal_eval(change_points)
    mask = np.zeros(length, dtype=np.int8)
    for idx in change_points:
        if 0 <= idx < length:
            mask[idx] = 1
    return mask.tolist()

In [32]:
# --- Process Change Points ---
# Add no of change points detected
change_points_df['num_change_points'] = change_points_df['change_points'].apply(len)
change_points_df['change_points'] = change_points_df['change_points'].apply(change_point_mask)

In [33]:
# --- Process Metadata ---
# Keep only selected columns
metadata_df = metadata_df[[
    'video_id', 'publishDate', 'duration_sec', 'viewCount', 'likeCount',
    'channelViewCount', 'subscriberCount', 'channelVideoCount']]

# Compute days since published (from 2025-07-01)
metadata_df['publishDate'] = pd.to_datetime(metadata_df['publishDate'], errors='coerce').dt.tz_localize(None)
reference_date = datetime(2025, 7, 1)
metadata_df['days_published'] = (reference_date - metadata_df['publishDate']).dt.days
metadata_df.drop(columns=['publishDate'], inplace=True)

# --- Merge all dataframes on video_id ---
df = emotions_df.merge(change_points_df, on="video_id", how="inner")
df = df.merge(metadata_df, on="video_id", how="inner")

In [34]:
df

,video_id,admiration,amusement,anger,annoyance,approval,caring,confusion,curiosity,desire,...,neutral,change_points,num_change_points,duration_sec,viewCount,likeCount,channelViewCount,subscriberCount,channelVideoCount,days_published
0,wL8qX260NTk,"[0.7320703268051147, 0.5988654000590545, 0.468...","[0.3544187843799591, 0.26652273475521737, 0.18...","[0.165129467844963, 0.1971059975901035, 0.2280...","[0.3012798130512237, 0.4159526021191568, 0.525...","[0.6537723541259766, 0.6795488945161453, 0.703...","[0.7575482130050659, 0.7378989096843835, 0.719...","[0.6837587356567383, 0.7505427598953247, 0.814...","[0.722643256187439, 0.7603157397472498, 0.7978...","[0.6324456334114075, 0.5751241174611178, 0.518...",...,"[0.5023497939109802, 0.5143594374560346, 0.525...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...",23,2204.0,64.0,3.0,1.165000e+04,60.0,72.0,422
1,I61WP51sSsc,"[0.3565481007099151, 0.4819501551112743, 0.546...","[0.405789703130722, 0.39608598146775753, 0.275...","[0.4664967954158783, 0.22705127283780258, 0.29...","[0.6357131004333496, 0.2717819262032556, 0.281...","[0.5273441076278687, 0.5557772137902, 0.704096...","[0.5393697023391724, 0.739505759393326, 0.7614...","[0.8535972237586975, 0.43476411128284953, 0.42...","[0.7244148254394531, 0.6635747004036952, 0.625...","[0.5913254022598267, 0.6780235442248258, 0.724...",...,"[0.3064590394496918, 0.4766763150691986, 0.457...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...",31,3266.0,10.0,1.0,4.490000e+02,28.0,17.0,27
2,bMmAI0neLaY,"[0.5921494364738464, 0.5158756465622873, 0.493...","[0.2679812014102936, 0.2649327656536391, 0.300...","[0.1929247975349426, 0.32299857880129956, 0.29...","[0.1944820433855056, 0.3054183911193501, 0.341...","[0.6917266249656677, 0.6311122142907345, 0.628...","[0.8326051235198975, 0.7927403016523882, 0.784...","[0.334815502166748, 0.41574292200984375, 0.530...","[0.5681260228157043, 0.6570314584356366, 0.675...","[0.6266080141067505, 0.6663212451067838, 0.616...",...,"[0.4407640993595123, 0.4385685369823918, 0.442...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...",19,5376.0,96.0,0.0,1.042000e+03,70.0,13.0,206
3,t58X0md283Y,"[0.6518335938453674, 0.5617997345298228, 0.532...","[0.2449305206537246, 0.31089595773003315, 0.34...","[0.2737078070640564, 0.23589130739370978, 0.21...","[0.3551653325557709, 0.3451310691207346, 0.361...","[0.5851302742958069, 0.6553265741377166, 0.585...","[0.7640964984893799, 0.7548317379421658, 0.760...","[0.6763123869895935, 0.6538849581371654, 0.602...","[0.7012510895729065, 0.6514174956263917, 0.760...","[0.7083688974380493, 0.6566446146579704, 0.645...",...,"[0.4290013015270233, 0.4660250755271526, 0.436...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",31,3483.0,128.0,2.0,4.811580e+05,3730.0,821.0,300
4,o8NiE3XMPrM,"[0.358776330947876, 0.5776983824643221, 0.6575...","[0.4788199663162231, 0.2699621563607996, 0.289...","[0.3236141800880432, 0.21810233728452158, 0.24...","[0.5772925019264221, 0.3094300546429374, 0.237...","[0.4718894064426422, 0.709445671601729, 0.6787...","[0.4341844320297241, 0.739007754759355, 0.7850...","[0.756392776966095, 0.5798122611912814, 0.2636...","[0.562288224697113, 0.6461373459209095, 0.6481...","[0.4614838063716888, 0.71097562529824, 0.63877...",...,"[0.4908329546451568, 0.4339071458036249, 0.371...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",29,6996.0,6875161.0,40845.0,4.628032e+09,13200000.0,2332.0,41
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25987,cc-ZTaw4Phk,"[0.3902280032634735, 0.4219332548102947, 0.453...","[0.2283354103565216, 0.2602710471008763, 0.292...","[0.1556959152221679, 0.15163238271318294, 0.14...","[0.2954810261726379, 0.2485092279284891, 0.201...","[0.6221400499343872, 0.6469896267158817, 0.671...","[0.751033365726471, 0.7994206776522627, 0.8478...","[0.5047340989112854, 0.4046951065761874, 0.304...","[0.6417994499206543, 0.6231136544786319, 0.604...","[0.6847631335258484, 0.6609948060729287, 0.63

In [35]:
from sklearn.preprocessing import RobustScaler
import pickle

feature_columns_to_scale = ['channelViewCount', 'subscriberCount', 'channelVideoCount']
target_columns_to_scale = ['viewCount', 'likeCount']

feature_scaler = RobustScaler()
feature_scaler.fit(df[feature_columns_to_scale])
with open('../models/feature_scaler.pkl', 'wb') as file:
    pickle.dump(feature_scaler, file) 

target_scaler = RobustScaler()
target_scaler.fit(df[target_columns_to_scale])
with open('../models/target_scaler.pkl', 'wb') as file:
    pickle.dump(target_scaler, file)

In [36]:
df[feature_columns_to_scale] = feature_scaler.transform(df[feature_columns_to_scale])
df[target_columns_to_scale] = target_scaler.transform(df[target_columns_to_scale])

In [37]:
df

,video_id,admiration,amusement,anger,annoyance,approval,caring,confusion,curiosity,desire,...,neutral,change_points,num_change_points,duration_sec,viewCount,likeCount,channelViewCount,subscriberCount,channelVideoCount,days_published
0,wL8qX260NTk,"[0.7320703268051147, 0.5988654000590545, 0.468...","[0.3544187843799591, 0.26652273475521737, 0.18...","[0.165129467844963, 0.1971059975901035, 0.2280...","[0.3012798130512237, 0.4159526021191568, 0.525...","[0.6537723541259766, 0.6795488945161453, 0.703...","[0.7575482130050659, 0.7378989096843835, 0.719...","[0.6837587356567383, 0.7505427598953247, 0.814...","[0.722643256187439, 0.7603157397472498, 0.7978...","[0.6324456334114075, 0.5751241174611178, 0.518...",...,"[0.5023497939109802, 0.5143594374560346, 0.525...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...",23,2204.0,-0.010763,-0.005487,-0.029160,-0.032785,-0.297045,422
1,I61WP51sSsc,"[0.3565481007099151, 0.4819501551112743, 0.546...","[0.405789703130722, 0.39608598146775753, 0.275...","[0.4664967954158783, 0.22705127283780258, 0.29...","[0.6357131004333496, 0.2717819262032556, 0.281...","[0.5273441076278687, 0.5557772137902, 0.704096...","[0.5393697023391724, 0.739505759393326, 0.7614...","[0.8535972237586975, 0.43476411128284953, 0.42...","[0.7244148254394531, 0.6635747004036952, 0.625...","[0.5913254022598267, 0.6780235442248258, 0.724...",...,"[0.3064590394496918, 0.4766763150691986, 0.457...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...",31,3266.0,-0.019989,-0.016461,-0.031129,-0.033349,-0.382582,27
2,bMmAI0neLaY,"[0.5921494364738464, 0.5158756465622873, 0.493...","[0.2679812014102936, 0.2649327656536391, 0.300...","[0.1929247975349426, 0.32299857880129956, 0.29...","[0.1944820433855056, 0.3054183911193501, 0.341...","[0.6917266249656677, 0.6311122142907345, 0.628...","[0.8326051235198975, 0.7927403016523882, 0.784...","[0.334815502166748, 0.41574292200984375, 0.530...","[0.5681260228157043, 0.6570314584356366, 0.675...","[0.6266080141067505, 0.6663212451067838, 0.616...",...,"[0.4407640993595123, 0.4385685369823918, 0.442...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...",19,5376.0,-0.005296,-0.021948,-0.031025,-0.032609,-0.388802,206
3,t58X0md283Y,"[0.6518335938453674, 0.5617997345298228, 0.532...","[0.2449305206537246, 0.31089595773003315, 0.34...","[0.2737078070640564, 0.23589130739370978, 0.21...","[0.3551653325557709, 0.3451310691207346, 0.361...","[0.5851302742958069, 0.6553265741377166, 0.585...","[0.7640964984893799, 0.7548317379421658, 0.760...","[0.6763123869895935, 0.6538849581371654, 0.602...","[0.7012510895729065, 0.6514174956263917, 0.760...","[0.7083688974380493, 0.6566446146579704, 0.645...",...,"[0.4290013015270233, 0.4660250755271526, 0.436...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",31,3483.0,0.000171,-0.010974,0.053359,0.031904,0.867807,300
4,o8NiE3XMPrM,"[0.358776330947876, 0.5776983824643221, 0.6575...","[0.4788199663162231, 0.2699621563607996, 0.289...","[0.3236141800880432, 0.21810233728452158, 0.24...","[0.5772925019264221, 0.3094300546429374, 0.237...","[0.4718894064426422, 0.709445671601729, 0.6787...","[0.4341844320297241, 0.739007754759355, 0.7850...","[0.756392776966095, 0.5798122611912814, 0.2636...","[0.562288224697113, 0.6461373459209095, 0.6481...","[0.4614838063716888, 0.71097562529824, 0.63877...",...,"[0.4908329546451568, 0.4339071458036249, 0.371...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",29,6996.0,1174.566950,224.093278,813.373930,232.634974,3.217729,41
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25987,cc-ZTaw4Phk,"[0.3902280032634735, 0.4219332548102947, 0.453...","[0.2283354103565216, 0.2602710471008763, 0.292...","[0.1556959152221679, 0.15163238271318294, 0.14...","[0.2954810261726379, 0.2485092279284891, 0.201...","[0.6221400499343872, 0.6469896267158817, 0.671...","[0.751033365726471, 0.7994206776522627, 0.8478...","[0.5047340989112854, 0.4046951065761874, 0.304...","[0.6417994499206543, 0.62311

In [38]:
print(target_scaler.inverse_transform([[1174.566950, 224.093278]]))

[[6875161.0000875   40844.9999155]]


In [39]:
# --- Save Final Dataset ---
df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved merged dataset to {OUTPUT_FILE} with {len(df)} rows")

Saved merged dataset to ../data/processed/dataset.csv with 25992 rows
